# Inspect Pitch Predictor Outputs

@hanoihantrakul 7JUN2024
If you are looking at this notebook after JUN2024 I have reduced the filesize by running cells that plot a lot of images and audio with incorrect variables so that the images/audio will not be saved in file. Just run the notebook in order and these errors will disappear.

@hanoihantrakul 23MAY2024
We want to use a perceptual pitch predictor as another loss signal when training the tokenizer. Before I use the pitch predictor this way, I want to make sure the predictions are accurate on dataset 2255 (mixed ZHEN vocal music) and 2375 (Speech and Karaoke data). This is because the original pitch predictor was only trained on 1038 (MSS data vocal music).

In [190]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [266]:
import torch
import torchaudio
import IPython.display as ipd
assert torch.cuda.is_available()
import sys
import os
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt

In [192]:
os.chdir('/opt/tiger/samantha')

## Load Pitch Predictor

In [193]:
from recipes.umm.requires.model_initializer import init_perceptual_pitch_predictor, init_rmvpe
from recipes.umm.models.voc_modules.pitch_predictor.inference import PerceptualPitchPredictor
from recipes.umm.models.rmvpe.inference import RMVPE
from recipes.umm.models.voc_modules.pitch_predictor import pitch_utils

In [363]:
MODELS_DICT = {
    "vocal_music": "hdfs://haruna/home/byte_speech_sv/hanoi.hantrakul/logs/umm_dual/pitch_predictor_trained_on_vocal/models_for_sharing/perceptual_pitch_model_1_17_2024.pt",
    "full_mix": "hdfs://haruna/home/byte_speech_sv/hanoi.hantrakul/logs/umm_dual/pitch_predictor_trained_on_full_mix/models_for_sharing/perceptual_pitch_model_1_26_2024.pt",
    "rmvpe": "hdfs://haruna/home/byte_speech_sv/user/chenyuanzhe/share/rmvpe.pt"
}

In [364]:
# Load Pitch Predictor trained on vocals MSS
pd_vocals = PerceptualPitchPredictor()
pd_vocals.load_and_eval(init_perceptual_pitch_predictor(MODELS_DICT["vocal_music"], local_rank=0, cache_dir=f"./.vocal_music")["state_dict"])

Succesfully loaded Perceptual Pitch Predictor!


In [365]:
# Load Pitch Predictor trained on Full Mix
pd_fullmix = PerceptualPitchPredictor()
pd_fullmix.load_and_eval(init_perceptual_pitch_predictor(MODELS_DICT["full_mix"], local_rank=0, cache_dir=f"./.full_mix")["state_dict"])

Succesfully loaded Perceptual Pitch Predictor!


In [366]:
# Load Pitch Predictor that is the original open source RMVPE model
pd_rmvpe = RMVPE()
pd_rmvpe.load_and_eval(init_rmvpe(MODELS_DICT["rmvpe"], local_rank=0, cache_dir=f"./.rmvpe")["state_dict"])

Load RMVPE successed!


## Load Data 1038 (MSS)

In [445]:
from recipes.datasets.mcc.mix_mkii import MixDataModule
from recipes.datasets.mcc.mix_mkii import MixMSSDataModule

In [1]:
# see
# samantha/recipes/umm/conf/perceptual_pitch_model/train_perceptual_pitch_model.yaml
SAMPLE_RATE = 24000
HOP_LENGTH = 240
BATCH_SIZE = 8
SHUFFLE_BUFFER_SIZE = 10
MIN_DURATION = 10
MAX_DURATION = 240
TOKENIZER = None
FRAME_RATE = 25

DATA_IDS = [1038]
DATA_WEIGHTS = [1]

mdm = MixMSSDataModule(
    data_ids=DATA_IDS,
    data_weights=DATA_WEIGHTS,
    batch_size=BATCH_SIZE * MAX_DURATION * SAMPLE_RATE, # means "total number of audio samples loaded into memory"
    shuffle_buffer_size=SHUFFLE_BUFFER_SIZE,
    sample_rate=SAMPLE_RATE,
    min_duration=MIN_DURATION,
    max_duration=MAX_DURATION,
    frame_rate=FRAME_RATE,
    tokenizer=TOKENIZER,
)

# There will be many printouts if loading for the first time

NameError: name 'MixMSSDataModule' is not defined

In [2]:
dm = mdm.train_dataloader()
dm_it = iter(dm)

NameError: name 'mdm' is not defined

In [448]:
NUM_ITERATIONS=1

audio_list = []
audio_vocal_list = []
audio_inst_list = []
token_list = []
for i in range(NUM_ITERATIONS):
    batch = next(dm_it)
    print(batch.keys())
    audio_list.append(batch['audio'])
    audio_vocal_list.append(batch['audio_vocal'])
    audio_inst_list.append(batch['audio_inst'])
    token_list.append(batch['token'])

dict_keys(['audio', 'audio_vocal', 'audio_inst', 'token'])


In [449]:
print(len(audio_list))
print(audio_list[0].shape)

1
torch.Size([80, 1, 575400])


In [450]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [451]:
pd_vocals_output = pd_vocals.forward_from_audio(audio_vocal_list[0].squeeze().to(device))
print(pd_vocals_output.shape)
f0_vocals = pd_vocals_output[:,:,0]
vuv_vocals = pd_vocals_output[:,:,1]

torch.Size([80, 2398, 2])


In [452]:
pd_fullmix_output = pd_fullmix.forward_from_audio(audio_list[0].squeeze().to(device))
print(pd_fullmix_output.shape)
f0_fullmix = pd_fullmix_output[:,:,0]
vuv_fullmix = pd_fullmix_output[:,:,1]

torch.Size([80, 2398, 2])


In [453]:
vocal_mel_input = pd_vocals.get_mel_spectrogram(audio_vocal_list[0].squeeze().to(device)) # input vocal mss audio to pd_vocal
print(vocal_mel_input.shape)

torch.Size([80, 2398, 160])


In [454]:
fullmix_mel_input = pd_fullmix.get_mel_spectrogram(audio_list[0].squeeze().to(device)) # input fullmix audio to pd_fullmix
print(fullmix_mel_input.shape)

torch.Size([80, 2398, 160])


In [455]:
def get_vuv(f0):
    vuv = f0.clone()
    vuv[vuv != 0] = 1
    return vuv

def compute_ground_truth_pitch(rmvpe_model, audio_waveform, sample_rate):
    """Use reference RMVPE model to get ground truth labels and follow the same post-processing steps as the lit_module."""
    gt_out = rmvpe_model.batch_infer(audio_waveform, sample_rate, thred=0.03, use_viterbi=False)
    f0 = gt_out
    vuv = get_vuv(f0)
    f0 = torch.log1p(f0) # No normalization. Only log1p for stability.
    return f0, vuv

In [456]:
f0_rmvpe_on_vocals, vuv_rmvpe_on_vocals = compute_ground_truth_pitch(pd_rmvpe, audio_vocal_list[0].squeeze().to(device), SAMPLE_RATE)
print(f0_rmvpe_on_vocals.shape)
print(vuv_rmvpe_on_vocals.shape)

torch.Size([80, 2398])
torch.Size([80, 2398])


In [457]:
SAMPLE_IDX = 70

In [3]:
# Full Mix Audio
ipd.Audio(audio_list[0][SAMPLE_IDX], rate=SAMPLE_RATE)

NameError: name 'ipd' is not defined

In [4]:
# Vocal MSS Audio
ipd.Audio(audio_vocal_list[0][SAMPLE_IDX], rate=SAMPLE_RATE)

NameError: name 'ipd' is not defined

In [5]:
_ = pitch_utils._create_fig_mel_with_f0_pred_and_gt(mel=vocal_mel_input[SAMPLE_IDX].cpu().numpy(),
                                                    f0_pred=pitch_utils.mask_f0_by_vuv(f0_vocals[SAMPLE_IDX].cpu().numpy(), pitch_utils.post_process_vuv_logits_to_binary_state(vuv_vocals[SAMPLE_IDX].cpu().numpy())),
                                                    #f0_pred=f0_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    f0_gt=f0_rmvpe_on_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    #vuv_pred=pitch_utils.post_process_vuv_logits_to_binary_state(vuv_vocals[SAMPLE_IDX].cpu().numpy()),
                                                    vuv_pred=None,
                                                    #vuv_gt=vuv_rmvpe_on_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    vuv_gt=None,
                                                    plot_title="Vocal MSS Pitch Predictor f0_hz with input: Vocal MSS vs Ground Truth RMVPE f0_hz with input: Vocal MSS")

NameError: name 'pitch_utils' is not defined

In [6]:
_ = pitch_utils._create_fig_mel_with_f0_pred_and_gt(mel=vocal_mel_input[SAMPLE_IDX].cpu().numpy(),
                                                    f0_pred=pitch_utils.mask_f0_by_vuv(f0_fullmix[SAMPLE_IDX].cpu().numpy(), pitch_utils.post_process_vuv_logits_to_binary_state(vuv_fullmix[SAMPLE_IDX].cpu().numpy())),
                                                    #f0_pred=f0_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    f0_gt=f0_rmvpe_on_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    #vuv_pred=pitch_utils.post_process_vuv_logits_to_binary_state(vuv_vocals[SAMPLE_IDX].cpu().numpy()),
                                                    vuv_pred=None,
                                                    #vuv_gt=vuv_rmvpe_on_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    vuv_gt=None,
                                                    plot_title="Full Mix Pitch Predictor f0_hz with input: Full Mix vs Ground Truth RMVPE f0_hz with input: Vocal MSS")

NameError: name 'pitch_utils' is not defined

In [7]:
f0_rmvpe_on_fullmix, vuv_rmvpe_on_fullmix = compute_ground_truth_pitch(pd_rmvpe, audio_list[0].squeeze().to(device), SAMPLE_RATE) # input fullmix to RMVPE
print(f0_rmvpe_on_fullmix.shape)
print(vuv_rmvpe_on_fullmix.shape)

NameError: name 'compute_ground_truth_pitch' is not defined

In [8]:
_ = pitch_utils._create_fig_mel_with_f0_pred_and_gt(mel=fullmix_mel_input[SAMPLE_IDX].cpu().numpy(),
                                                    f0_pred=pitch_utils.mask_f0_by_vuv(f0_fullmix[SAMPLE_IDX].cpu().numpy(), pitch_utils.post_process_vuv_logits_to_binary_state(vuv_fullmix[SAMPLE_IDX].cpu().numpy())),
                                                    #f0_pred=f0_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    f0_gt=f0_rmvpe_on_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    #vuv_pred=pitch_utils.post_process_vuv_logits_to_binary_state(vuv_vocals[SAMPLE_IDX].cpu().numpy()),
                                                    vuv_pred=None,
                                                    #vuv_gt=vuv_rmvpe[SAMPLE_IDX].cpu().numpy(),
                                                    vuv_gt=None,
                                                    plot_title="Full Mix Pitch Predictor f0_hz with input: Full Mix vs Ground Truth RMVPE f0_hz with input: Vocal MSS")

NameError: name 'pitch_utils' is not defined

In [9]:
_ = pitch_utils._create_fig_mel_with_f0_pred_and_gt(mel=fullmix_mel_input[SAMPLE_IDX].cpu().numpy(),
                                                    f0_pred=f0_rmvpe_on_fullmix[SAMPLE_IDX].cpu().numpy(),
                                                    #f0_pred=f0_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    f0_gt=f0_rmvpe_on_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    #vuv_pred=pitch_utils.post_process_vuv_logits_to_binary_state(vuv_vocals[SAMPLE_IDX].cpu().numpy()),
                                                    vuv_pred=None,
                                                    #vuv_gt=vuv_rmvpe[SAMPLE_IDX].cpu().numpy(),
                                                    vuv_gt=None,
                                                    plot_title="RMVPE f0_hz with input: Full Mix vs Ground Truth RMVPE f0_hz with input: Vocal MSS")

NameError: name 'pitch_utils' is not defined

In [10]:
pd_fullmix_hidden_state = pd_fullmix.get_hidden_state()
pd_vocals_hidden_state = pd_vocals.get_hidden_state()
print(pd_fullmix_hidden_state.shape)
print(pd_vocals_hidden_state.shape)

NameError: name 'pd_fullmix' is not defined

In [11]:
def create_fig_hidden_state_with_f0(
    mel,
    f0_pred,
    convert_to_raw_hz=True,
    mel_bins_max=160,
    f0_hz_max=800,
    plot_title=None,
):
    """Plot f0_pred and f0_gt ontop of mel spectrogram. TODO: handle np vs torch. This function expects numpy."""
    # Configure plot
    fig = plt.figure(figsize=(16, 8))
    legend_text = []
    # x axis time markings
    t = np.arange(0, len(f0_pred), 1)
    # Plot Mel
    plt.pcolor(mel.T)
    legend_text.append("Hidden State")
    # Plot Raw F0_Hz Pred
    f0_hz_pred = np.expm1(f0_pred) if convert_to_raw_hz else f0_pred
    f0_hz_pred = (
        f0_hz_pred / f0_hz_max
    ) * mel_bins_max  # scales the f0 values to fit on the mel spectrogram plot.
    plt.plot(t, f0_hz_pred, "red", linewidth=2)
    legend_text.append("F0_hz PitchPredictor")
    # # Plot Raw F0_Hz GT
    # f0_hz_gt = np.expm1(f0_gt) if convert_to_raw_hz else f0_gt
    # f0_hz_gt = (
    #     f0_hz_gt / f0_hz_max
    # ) * mel_bins_max  # scales the f0 values to fit on the mel spectrogram plot.
    # plt.plot(t, f0_hz_gt, "blue", linewidth=2)
    # legend_text.append("F0_hz RMVPE on Full Mix")
       # Titles and legends
    plt.legend(legend_text)
    if plot_title is not None:
        plt.title(plot_title)
    else:
        plt.title("Combined Mel spectrogram, Pred Raw F0, GT Raw F0 and VUV")
    return fig

In [12]:
i = 70
print(np.max(pd_vocals_hidden_state[i].T.cpu().numpy()))
print(np.min(pd_vocals_hidden_state[i].T.cpu().numpy()))
print(np.mean(pd_vocals_hidden_state[i].T.cpu().numpy()))
#plt.pcolor(pd_vocals_hidden_state[i].T.cpu().numpy(), vmin=-15, vmax=15)
pd_vocals_processed_f0 = pitch_utils.mask_f0_by_vuv(f0_vocals[i].cpu().numpy(), pitch_utils.post_process_vuv_logits_to_binary_state(vuv_vocals[i].cpu().numpy()))
create_fig_hidden_state_with_f0(mel=pd_vocals_hidden_state[i].cpu().numpy(), 
                                f0_pred=pd_vocals_processed_f0,
                                mel_bins_max=128,
                                plot_title="128 dim hidden state and f0_hz")

NameError: name 'np' is not defined

In [13]:
# Full Mix Audio
ipd.Audio(audio_vocal_list[0][i], rate=SAMPLE_RATE)

NameError: name 'ipd' is not defined

In [14]:
print(np.max(pd_fullmix_hidden_state[i].T.cpu().numpy()))
print(np.min(pd_fullmix_hidden_state[i].T.cpu().numpy()))
print(np.mean(pd_fullmix_hidden_state[i].T.cpu().numpy()))
#plt.pcolor(pd_vocals_hidden_state[i].T.cpu().numpy(), vmin=-15, vmax=15)
pd_fullmix_processed_f0 = pitch_utils.mask_f0_by_vuv(f0_fullmix[i].cpu().numpy(), pitch_utils.post_process_vuv_logits_to_binary_state(vuv_fullmix[i].cpu().numpy()))
create_fig_hidden_state_with_f0(mel=pd_fullmix_hidden_state[i].cpu().numpy(), 
                                f0_pred=pd_fullmix_processed_f0,
                                mel_bins_max=128,
                                plot_title="128 dim hidden state and f0_hz")

NameError: name 'np' is not defined

In [495]:
# Full Mix Audio
ipd.Audio(audio_list[0][i], rate=SAMPLE_RATE)

# Data 2255

In [15]:
# see
# samantha/recipes/umm/conf/convumm_gan/convumm_gan_719M_25hz_vocals_2255_2375.yaml
SAMPLE_RATE = 24000
HOP_LENGTH = 240
BATCH_SIZE = 8
SHUFFLE_BUFFER_SIZE = 10
MIN_DURATION = 1
MAX_DURATION = 60
TOKENIZER = None
FRAME_RATE = 25

DATA_IDS = [2255]
DATA_WEIGHTS = [1]

mdm = MixDataModule(
    data_ids=DATA_IDS,
    data_weights=DATA_WEIGHTS,
    batch_size=BATCH_SIZE * MAX_DURATION * SAMPLE_RATE, # means "total number of audio samples loaded into memory"
    shuffle_buffer_size=SHUFFLE_BUFFER_SIZE,
    sample_rate=SAMPLE_RATE,
    min_duration=MIN_DURATION,
    max_duration=MAX_DURATION,
    frame_rate=FRAME_RATE,
    tokenizer=TOKENIZER,
)

# There will be many printouts if loading for the first time

NameError: name 'MixDataModule' is not defined

In [16]:
dm = mdm.train_dataloader()
dm_it = iter(dm)

NameError: name 'mdm' is not defined

In [551]:
NUM_ITERATIONS=1

audio_list = []
token_list = []
for i in range(NUM_ITERATIONS):
    batch = next(dm_it)
    print(batch.keys())
    audio_list.append(batch['audio'])
    token_list.append(batch['token'])

dict_keys(['audio', 'token'])


In [552]:
print(len(audio_list))
print(audio_list[0].shape)

1
torch.Size([10, 1, 1092312])


In [553]:
pd_fullmix_output = pd_fullmix.forward_from_audio(audio_list[0].squeeze().to(device))
print(pd_fullmix_output.shape)
f0_fullmix = pd_fullmix_output[:,:,0]
vuv_fullmix = pd_fullmix_output[:,:,1]

pd_fullmix_hidden_state = pd_fullmix.get_hidden_state()
print(pd_fullmix_hidden_state.shape)

f0_rmvpe_on_fullmix, vuv_rmvpe_on_fullmix = compute_ground_truth_pitch(pd_rmvpe, audio_list[0].squeeze().to(device), SAMPLE_RATE) # input fullmix to RMVPE
print(f0_rmvpe_on_fullmix.shape)
print(vuv_rmvpe_on_fullmix.shape)

torch.Size([10, 4552, 2])
torch.Size([10, 4552, 128])
torch.Size([10, 4552])
torch.Size([10, 4552])


In [554]:
fullmix_mel_input = pd_fullmix.get_mel_spectrogram(audio_list[0].squeeze().to(device)) # input fullmix audio to pd_fullmix
print(fullmix_mel_input.shape)

torch.Size([10, 4552, 160])


In [17]:
SAMPLE_IDX=8
# Full Mix Audio
ipd.Audio(audio_list[0][SAMPLE_IDX], rate=SAMPLE_RATE)

NameError: name 'ipd' is not defined

In [18]:
def create_fig_mel_with_f0_from_two_competing_models(
    mel,
    f0_pred,
    f0_gt,
    vuv_pred=None,
    vuv_gt=None,
    convert_to_raw_hz=True,
    mel_bins_max=160,
    f0_hz_max=800,
    plot_title=None,
):
    """Plot f0_pred and f0_gt ontop of mel spectrogram. TODO: handle np vs torch. This function expects numpy."""
    # Configure plot
    fig = plt.figure(figsize=(16, 8))
    legend_text = []
    # x axis time markings
    t = np.arange(0, len(f0_pred), 1)
    # Plot Mel
    plt.pcolor(mel.T, vmin=-6, vmax=0.5)
    legend_text.append("Mel")
    # Plot Raw F0_Hz Pred
    f0_hz_pred = np.expm1(f0_pred) if convert_to_raw_hz else f0_pred
    f0_hz_pred = (
        f0_hz_pred / f0_hz_max
    ) * mel_bins_max  # scales the f0 values to fit on the mel spectrogram plot.
    plt.plot(t, f0_hz_pred, "red", linewidth=2)
    legend_text.append("F0_hz PitchPredictorFullMix")
    # Plot Raw F0_Hz GT
    f0_hz_gt = np.expm1(f0_gt) if convert_to_raw_hz else f0_gt
    f0_hz_gt = (
        f0_hz_gt / f0_hz_max
    ) * mel_bins_max  # scales the f0 values to fit on the mel spectrogram plot.
    plt.plot(t, f0_hz_gt, "blue", linewidth=2)
    legend_text.append("F0_hz RMVPE on Full Mix")
       # Titles and legends
    plt.legend(legend_text)
    if plot_title is not None:
        plt.title(plot_title)
    else:
        plt.title("Combined Mel spectrogram, Pred Raw F0, GT Raw F0 and VUV")
    return fig

In [19]:
_ = create_fig_mel_with_f0_from_two_competing_models(mel=fullmix_mel_input[SAMPLE_IDX].cpu().numpy(),
                                                    f0_pred=pitch_utils.mask_f0_by_vuv(f0_fullmix[SAMPLE_IDX].cpu().numpy(), pitch_utils.post_process_vuv_logits_to_binary_state(vuv_fullmix[SAMPLE_IDX].cpu().numpy())),
                                                    #f0_pred=f0_vocals[SAMPLE_IDX].cpu().numpy(),
                                                    f0_gt=f0_rmvpe_on_fullmix[SAMPLE_IDX].cpu().numpy(),
                                                    #vuv_pred=pitch_utils.post_process_vuv_logits_to_binary_state(vuv_vocals[SAMPLE_IDX].cpu().numpy()),
                                                    vuv_pred=None,
                                                    #vuv_gt=vuv_rmvpe[SAMPLE_IDX].cpu().numpy(),
                                                    vuv_gt=None,
                                                    plot_title="Full Mix Pitch Predictor f0_hz with input: Full Mix vs RMVPE f0_hz with input: Full Mix")

NameError: name 'fullmix_mel_input' is not defined

In [20]:
print(np.max(pd_fullmix_hidden_state[SAMPLE_IDX].T.cpu().numpy()))
print(np.min(pd_fullmix_hidden_state[SAMPLE_IDX].T.cpu().numpy()))
print(np.mean(pd_fullmix_hidden_state[SAMPLE_IDX].T.cpu().numpy()))
#plt.pcolor(pd_vocals_hidden_state[i].T.cpu().numpy(), vmin=-15, vmax=15)
pd_fullmix_processed_f0 = pitch_utils.mask_f0_by_vuv(f0_fullmix[SAMPLE_IDX].cpu().numpy(), pitch_utils.post_process_vuv_logits_to_binary_state(vuv_fullmix[SAMPLE_IDX].cpu().numpy()))
create_fig_hidden_state_with_f0(mel=pd_fullmix_hidden_state[SAMPLE_IDX].cpu().numpy(), 
                                f0_pred=pd_fullmix_processed_f0,
                                mel_bins_max=128,
                                plot_title="128 dim hidden state and f0_hz")

NameError: name 'np' is not defined